# 14a — local Phase3 data preparation

This notebook now runs locally; the old filename is retained so existing links still work. The cloud version is backed up outside the repository (see LOCAL_RUN.md).

School experiment: use the supplied Phase3 bundle, not the old testset. No installs, downloads, API calls or human-approval checkbox. Original bundle files and the original run remain unchanged. This notebook defaults to the small post-review wildfire amendment in runs/revision/reviewed/. Its method is in the plan; original repairs remain in review_decisions.json.


In [ ]:
from pathlib import Path
import sys
import json

# Works when launched from the repository, Phase3, or a directory below either.
locations = [Path.cwd(), *Path.cwd().parents]
PHASE3 = next((p if (p / "phase3_run.py").is_file() else p / "Phase3"
               for p in locations
               if (p / "phase3_run.py").is_file() or (p / "Phase3/phase3_run.py").is_file()), None)
if PHASE3 is None:
    raise RuntimeError("Open this notebook from the project or Phase3 directory.")
sys.path.insert(0, str(PHASE3))
from phase3_run import load_settings, preflight
CONFIG = "revision_config.json"  # local_config.json selects the original experiment
settings = load_settings(PHASE3 / CONFIG)
print("Kernel:", sys.executable)
print("Phase3:", PHASE3)


## 1. Rebuild the reviewed draft

This applies only recorded, evidence-checked repairs. It does not resample questions or certify unresolved labels.


In [ ]:
if CONFIG == "revision_config.json":
    from phase3_revision import prepare_revision
    report = prepare_revision()
else:
    from review_phase3 import prepare
    report = prepare(corpus_path=settings["corpus"])
print(json.dumps({k: report[k] for k in ("cases", "changed_cases", "content_status", "corpus_rows", "ready_for_generation")}, indent=2))


## 2. Check local retrieval setup

Choose the installed NewsQA Phase3 (local CUDA) kernel. The selected BGE dependencies and weights are downloaded; local_config.json uses this machine's GPU. See LOCAL_RUN.md for the explicit setup commands if recreating the environment. This check does not install anything.


In [ ]:
retrieval_check = preflight(settings, "retrieve")
print(json.dumps(retrieval_check, indent=2))


## 3. Collect the actual contexts when setup is available

BGE-M3 → top 20 → BGE-reranker-large → top 5. Explicit ablation/weak contexts retain their embedded texts. Natural misses are re-observed, not manufactured. Retrieval is cached locally. No Gemini requests are made here.


In [ ]:
COLLECT_RETRIEVALS = False
if COLLECT_RETRIEVALS:
    from phase3_run import collect_retrievals
    print(collect_retrievals(settings))
else:
    print("Retrieval is off. Inspect the setup report before enabling it.")


## 4. Evidence check before 14b

The fresh selected contexts must be inspected against each question, including aliases and false-premise corrections. This is an AI evidence review, not a request for human review. Known content holds remain visible in CASE_REVIEW.md; do not mark them accepted just to reach 200.

After actual retrieval inspection has resolved the labels, record evidence decisions in the selected work directory's retrieval_review.jsonl and freeze the experiment. The generated template contains pending entries, not approvals. The revision is explicitly post-review, not a new untouched test; its changed control must be checked before new generation. Unchanged requests can be reused through phase3_revision.py seed.


In [ ]:
FREEZE_REVIEWED_EXPERIMENT = False
if FREEZE_REVIEWED_EXPERIMENT:
    from phase3_run import freeze
    print(freeze(settings))
else:
    print("No dataset was frozen or approved by executing this cell.")
